In [4]:
# Minimal pose stream using the simple startMission overload (no ClientPool)
import json, time, os, shutil
from malmo.MalmoPython import AgentHost, MissionSpec, MissionRecordSpec, ClientPool, ClientInfo
client_pool = ClientPool()
client_pool.add(ClientInfo("127.0.0.1", 10000))
MISSION = """<?xml version="1.0" encoding="UTF-8" standalone="no" ?>
<Mission xmlns="http://ProjectMalmo.microsoft.com" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">

  <About>
    <Summary>Extracting camera pose and frames for later SLAM</Summary>
  </About>

  <ServerSection>
    <ServerInitialConditions>
            <Time>
                <StartTime>3000</StartTime>
                <AllowPassageOfTime>false</AllowPassageOfTime>
            </Time>
            <Weather>clear</Weather>
            <AllowSpawning>false</AllowSpawning>
    </ServerInitialConditions>
    
    <ServerHandlers>
      <FlatWorldGenerator generatorString="3;1*minecraft:dirt,1*minecraft:grass;2;village" forceReset="true"/>
      
      <DrawingDecorator>
        <!-- Side walls -->
        <DrawCuboid x1="-2" y1="2" z1="-1" x2="-2" y2="5" z2="10" type="stone"/>
        <DrawCuboid x1="2"  y1="2" z1="-1" x2="2"  y2="5" z2="7" type="stone"/>
        
        <!-- Back wall -->
        <DrawCuboid x1="-2" y1="2" z1="-2" x2="2" y2="5" z2="-2" type="stone"/>
        
        <!-- Left turn -->
        <DrawCuboid x1="-2" y1="2" z1="11" x2="7" y2="5" z2="11" type="stone"/>
        <DrawCuboid x1="2"  y1="2" z1="8" x2="7"  y2="5" z2="8" type="stone"/>

        <!-- Left turn wall -->
        <DrawCuboid x1="8" y1="2" z1="8" x2="8" y2="5" z2="11" type="stone"/>
        
        <!-- Ceiling -->
        <DrawCuboid x1="-2" y1="5" z1="-1" x2="2"  y2="5" z2="11" type="stone"/>
        <DrawCuboid x1="2" y1="5" z1="8" x2="8"  y2="5" z2="11" type="stone"/>

        <!-- Light-->
        <DrawBlock x="-1"  y="2" z="-1" type="torch"/>
        <DrawBlock x="-1"  y="2" z="2" type="torch"/>
        <DrawBlock x="-1"  y="2" z="5" type="torch"/>
        <DrawBlock x="-1" y="2" z="7" type="torch"/>
        <DrawBlock x="-1" y="2" z="10" type="torch"/>
        <DrawBlock x="2" y="2" z="10" type="torch"/>
        <DrawBlock x="5" y="2" z="10" type="torch"/>
      </DrawingDecorator>
      
      <ServerQuitFromTimeUp description="" timeLimitMs="9500"/>
    </ServerHandlers>
    
  </ServerSection>

  <AgentSection mode="Survival">
    <Name>Amidouguis</Name>
    
    <AgentStart>
      <Placement pitch="0" x="0.0" y="2.0" yaw="0" z="0.0"/>
    </AgentStart>
    
    <AgentHandlers>
      <ContinuousMovementCommands turnSpeedDegs="180"/>
      
      <ObservationFromFullStats/>
      
      <VideoProducer want_depth="true">
        <Width>640</Width>
        <Height>360</Height>
      </VideoProducer>
    </AgentHandlers>
    
  </AgentSection>

</Mission>
"""

# Defining camera motions
def walk_forward(ah):
    ah.sendCommand("move 1")
    
def stop_walking(ah):
    ah.sendCommand("move 0")
    
def turn_left(ah):
    ah.sendCommand("turn -1")
    
def turn_right(ah):
    ah.sendCommand("turn 1")
    
def stop_turning(ah):
    ah.sendCommand("turn 0")

# Setting up folders
root_dir = "../data/small_corridor"
images_dir = os.path.join(root_dir, "images")
shutil.rmtree(root_dir, ignore_errors=True)
os.makedirs(images_dir, exist_ok=True)
pose_path = os.path.join(root_dir, "poses.txt")

# Instantiating agent and mission
ah = AgentHost()
ms = MissionSpec(MISSION, True)
mr = MissionRecordSpec("../data/small_corridor/run.tgz")
mr.recordMP4(20, 400000) 

print("Initializing mission")
ah.startMission(ms, mr)

# Mission start error handling
t0 = time.time()
while True:
    ws = ah.getWorldState()
    
    if ws.has_mission_begun:
        break
    
    if any(ws.errors):
        for e in ws.errors: 
            print("Error:", e.text)
        raise RuntimeError("Mission failed to start due to above errors.")
    
    if time.time() - t0 > 10:
        raise RuntimeError("Mission failed to start due to timeout.")
    
    time.sleep(0.1)

# Data extraction loop
frame_idx = 1
obs_counter = 0
init_time = 0
with open(pose_path, "w", encoding="utf-8") as pose_file:
    
    while ws.is_mission_running:
        ws = ah.getWorldState()

        last_pose = False
        last_frame = False
        
        if ws.observations:
            obs_counter += 1
            
            obs = json.loads(ws.observations[-1].text)
            x, y, z = float(obs.get("XPos")), float(obs.get("YPos")), float(obs.get("ZPos"))
            yaw, pitch = float(obs.get("Yaw")), float(obs.get("Pitch"))
            last_pose = True
            
            # Sending camera movements
            time_elapsed = float(obs.get("TotalTime", 0.0))
            time_elapsed = time_elapsed/20
            
            # Correcting time elapsed based on mission initialization time
            if obs_counter == 1:
                init_time = time_elapsed
            time_elapsed = time_elapsed - init_time
            
            if time_elapsed < 2.5:
                walk_forward(ah)
            elif time_elapsed < 3.0:
                stop_walking(ah)
                turn_left(ah)
            elif time_elapsed < 4.5:
                stop_turning(ah)
                walk_forward(ah)
            elif time_elapsed < 5.5:
                stop_walking(ah)
                turn_right(ah)
            elif time_elapsed < 6.9:
                stop_turning(ah)
                walk_forward(ah)
            elif time_elapsed < 7.4:
                stop_walking(ah)
                turn_right(ah)
            elif time_elapsed < 8.3:
                stop_turning(ah)
                walk_forward(ah)
            elif time_elapsed > 9.3:
                stop_walking(ah)
            
        if ws.video_frames:
            frame = ws.video_frames[-1]
            rgb_image_filename = "{:05d}_rgb.ppm".format(frame_idx)
            rgb_image_path = os.path.join(images_dir, rgb_image_filename)
            header = "P6\n{} {}\n255\n".format(frame.width, frame.height)
            d_image_filename = "{:05d}_d.pgm".format(frame_idx)
            d_image_path = os.path.join(images_dir, d_image_filename)
            d_header = "P5\n{} {}\n255\n".format(frame.width, frame.height)
            
            frame_idx += 1
            last_frame = True
                
        if last_pose and last_frame:
            pose_file.write("{:.4f},{:.4f},{:.4f},{:.4f},{:.4f},{}\n".format(x, y, z, yaw, pitch, rgb_image_path))
            rgb_bytes = bytearray()
            raw = bytes(frame.pixels)
            for i in range(0, len(raw), 4):
                rgb_bytes.extend(raw[i:i+3])
            with open(rgb_image_path, "wb") as f:
                f.write(header.encode("ascii"))
                f.write(rgb_bytes)
            if frame.channels == 4:
                depth_bytes = (bytearray(raw[3::4]))
                with open(d_image_path, "wb") as f:
                    f.write(d_header.encode("ascii"))
                    f.write(depth_bytes)

            
        time.sleep(0.01)
    
print("Finalizing mission")

Initializing mission
Finalizing mission
